# **Raw Baseline ELCS**

In [1]:
import pandas as pd
import numpy as np

from skeLCS.eLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

data_raw = pd.read_csv("../data/weatherAUS.csv")

print("Original shape:", data_raw.shape)
print(data_raw.head())


Original shape: (145460, 23)
         Date Location  MinTemp  MaxTemp  Rainfall  Evaporation  Sunshine  \
0  2008-12-01   Albury     13.4     22.9       0.6          NaN       NaN   
1  2008-12-02   Albury      7.4     25.1       0.0          NaN       NaN   
2  2008-12-03   Albury     12.9     25.7       0.0          NaN       NaN   
3  2008-12-04   Albury      9.2     28.0       0.0          NaN       NaN   
4  2008-12-05   Albury     17.5     32.3       1.0          NaN       NaN   

  WindGustDir  WindGustSpeed WindDir9am  ... Humidity9am  Humidity3pm  \
0           W           44.0          W  ...        71.0         22.0   
1         WNW           44.0        NNW  ...        44.0         25.0   
2         WSW           46.0          W  ...        38.0         30.0   
3          NE           24.0         SE  ...        45.0         16.0   
4           W           41.0        ENE  ...        82.0         33.0   

   Pressure9am  Pressure3pm  Cloud9am  Cloud3pm  Temp9am  Temp3pm  Ra

In [2]:
# remove rows where the target is missing
data_raw = data_raw.dropna(
    subset=["RainTomorrow"]
).copy()

# convert target to numeric
data_raw["RainTomorrow"] = data_raw["RainTomorrow"].map({
    "No": 0,
    "Yes": 1
})

# Date is removed for the raw baseline rather than feature engineered
data_raw = data_raw.drop(columns=["Date"])

print("Shape after removing missing target:", data_raw.shape)

print("\nTarget distribution:")
print(data_raw["RainTomorrow"].value_counts())

print("\nMissing feature values:")
print(data_raw.isna().sum().sort_values(ascending=False))


Shape after removing missing target: (142193, 22)

Target distribution:
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64

Missing feature values:
Sunshine         67816
Evaporation      60843
Cloud3pm         57094
Cloud9am         53657
Pressure9am      14014
Pressure3pm      13981
WindDir9am       10013
WindGustDir       9330
WindGustSpeed     9270
WindDir3pm        3778
Humidity3pm       3610
Temp3pm           2726
WindSpeed3pm      2630
Humidity9am       1774
RainToday         1406
Rainfall          1406
WindSpeed9am      1348
Temp9am            904
MinTemp            637
MaxTemp            322
Location             0
RainTomorrow         0
dtype: int64


In [3]:
classlabel = "RainTomorrow"

x_raw = data_raw.drop(columns=[classlabel])
y_raw = data_raw[classlabel]

x_train, x_test, y_train, y_test = train_test_split(
    x_raw,
    y_raw,
    test_size=0.20,
    random_state=42,
    stratify=y_raw
)

print("Training:", x_train.shape)
print("Testing:", x_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())


Training: (113754, 21)
Testing: (28439, 21)

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [4]:
# categorical values must be numeric for eLCS
categorical_cols = x_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

x_train = x_train.copy()
x_test = x_test.copy()

x_train[categorical_cols] = encoder.fit_transform(
    x_train[categorical_cols]
)

x_test[categorical_cols] = encoder.transform(
    x_test[categorical_cols]
)


Categorical columns:
['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [5]:
# tell eLCS which encoded columns are discrete categories
discrete_attributes = np.array([
    x_train.columns.get_loc(col)
    for col in categorical_cols
])

print("Discrete attribute indices:")
print(discrete_attributes)

# professor's eLCS requires numeric numpy arrays
x_train_np = x_train.to_numpy(dtype=float)
x_test_np = x_test.to_numpy(dtype=float)

y_train_np = y_train.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float)

print("\nTraining shape:", x_train_np.shape)
print("Testing shape:", x_test_np.shape)
print("Features:", x_train_np.shape[1])

print("\nMissing training feature values:",
      np.isnan(x_train_np).sum())

print("Missing testing feature values:",
      np.isnan(x_test_np).sum())


Discrete attribute indices:
[ 0  6  8  9 20]

Training shape: (113754, 21)
Testing shape: (28439, 21)
Features: 21

Missing training feature values: 253678
Missing testing feature values: 62881


In [6]:
# full raw eLCS baseline
# learning iterations match the number of training instances
raw_elcs = eLCS(
    learning_iterations=len(x_train_np),
    N=10000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

raw_elcs.fit(
    x_train_np,
    y_train_np
)

print("Raw eLCS training complete")

raw_predictions = raw_elcs.predict(
    x_test_np
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print(
    "Balanced Accuracy:",
    balanced_accuracy_score(
        y_test_np,
        raw_predictions
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_np,
        raw_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_np,
        raw_predictions
    )
)


Raw eLCS training complete
Accuracy: 0.8294595449910335
Balanced Accuracy: 0.6384023411395015

Classification Report:
              precision    recall  f1-score   support

         0.0       0.83      0.98      0.90     22064
         1.0       0.85      0.29      0.43      6375

    accuracy                           0.83     28439
   macro avg       0.84      0.64      0.67     28439
weighted avg       0.83      0.83      0.80     28439


Confusion Matrix:
[[21727   337]
 [ 4513  1862]]


In [ ]:
# ============================================================
# SAVE RAW eLCS PREDICTIONS
# ============================================================

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

raw_results = pd.DataFrame({
    "row_id": x_test.index.to_numpy(),
    "actual": y_test.to_numpy(),
    "prediction": raw_predictions
})

raw_results.to_csv(
    results_dir / "raw_elcs_predictions.csv",
    index=False
)

print("Raw eLCS predictions saved.")
print(raw_results.head())

Raw eLCS predictions saved


# **Creating the cleaned dataset**

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split


# ============================================================
# LOAD RAW DATA
# ============================================================

df_clean_source = pd.read_csv("../data/weatherAUS.csv")

print("Original shape:", df_clean_source.shape)


# ============================================================
# REMOVE MISSING TARGET
# RainTomorrow cannot be imputed because it is the class label
# ============================================================

df_clean_source = df_clean_source.dropna(
    subset=["RainTomorrow"]
).copy()

df_clean_source["RainTomorrow"] = (
    df_clean_source["RainTomorrow"]
    .astype(str)
    .str.strip()
    .map({
        "No": 0,
        "Yes": 1
    })
)

print("After removing missing target:", df_clean_source.shape)
print(df_clean_source["RainTomorrow"].value_counts())


# ============================================================
# SPLIT BEFORE LEARNED PREPROCESSING
# ============================================================

x_source = df_clean_source.drop(
    columns=["RainTomorrow"]
)

y_source = df_clean_source["RainTomorrow"].astype(int)

clean_x_train_raw, clean_x_test_raw, clean_y_train, clean_y_test = (
    train_test_split(
        x_source,
        y_source,
        test_size=0.20,
        random_state=42,
        stratify=y_source
    )
)

# separate copies so the original split remains available
clean_x_train = clean_x_train_raw.copy()
clean_x_test = clean_x_test_raw.copy()

print("\nTraining rows:", len(clean_x_train))
print("Testing rows:", len(clean_x_test))

print("\nTraining target:")
print(clean_y_train.value_counts())

print("\nTesting target:")
print(clean_y_test.value_counts())

Original shape: (145460, 23)
After removing missing target: (142193, 23)
RainTomorrow
0    110316
1     31877
Name: count, dtype: int64

Training rows: 113754
Testing rows: 28439

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [9]:
# ============================================================
# CLEAN CATEGORY FORMATTING
# ============================================================

clean_x_train = clean_x_train.copy()
clean_x_test = clean_x_test.copy()

categorical_columns = [
    "Location",
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
    "RainToday"
]

# strip accidental whitespace
for col in categorical_columns:
    clean_x_train[col] = clean_x_train[col].str.strip()
    clean_x_test[col] = clean_x_test[col].str.strip()


# ============================================================
# NUMERICAL IMPUTATION
# MEDIANS ARE LEARNED FROM TRAINING DATA ONLY
# ============================================================

numeric_columns = [
    "MinTemp",
    "MaxTemp",
    "Rainfall",
    "Evaporation",
    "Sunshine",
    "WindGustSpeed",
    "WindSpeed9am",
    "WindSpeed3pm",
    "Humidity9am",
    "Humidity3pm",
    "Pressure9am",
    "Pressure3pm",
    "Cloud9am",
    "Cloud3pm",
    "Temp9am",
    "Temp3pm"
]

train_medians = clean_x_train[numeric_columns].median()

for col in numeric_columns:
    clean_x_train[col] = clean_x_train[col].fillna(
        train_medians[col]
    )

    clean_x_test[col] = clean_x_test[col].fillna(
        train_medians[col]
    )


# ============================================================
# CATEGORICAL IMPUTATION
# MODES ARE LEARNED FROM TRAINING DATA ONLY
# ============================================================

categorical_fill_columns = [
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
    "RainToday"
]

train_modes = {}

for col in categorical_fill_columns:
    train_modes[col] = clean_x_train[col].mode()[0]

    clean_x_train[col] = clean_x_train[col].fillna(
        train_modes[col]
    )

    clean_x_test[col] = clean_x_test[col].fillna(
        train_modes[col]
    )


# ============================================================
# CHECK RESULTS
# ============================================================

print("Training medians:")
print(train_medians)

print("\nTraining modes:")
print(train_modes)

print("\nMissing values in training:")
print(clean_x_train.isna().sum().sum())

print("\nMissing values in testing:")
print(clean_x_test.isna().sum().sum())

Training medians:
MinTemp            12.0
MaxTemp            22.6
Rainfall            0.0
Evaporation         4.8
Sunshine            8.5
WindGustSpeed      39.0
WindSpeed9am       13.0
WindSpeed3pm       19.0
Humidity9am        70.0
Humidity3pm        52.0
Pressure9am      1017.6
Pressure3pm      1015.2
Cloud9am            5.0
Cloud3pm            5.0
Temp9am            16.7
Temp3pm            21.1
dtype: float64

Training modes:
{'WindGustDir': 'W', 'WindDir9am': 'N', 'WindDir3pm': 'SE', 'RainToday': 'No'}

Missing values in training:
0

Missing values in testing:
0


In [10]:
# ============================================================
# APPLY ESTABLISHED DATA CLEANING RULES
# ============================================================

# Cloud cover is measured from 0 to 8 oktas.
# Cap previously identified invalid values above 8.
for col in ["Cloud9am", "Cloud3pm"]:
    clean_x_train.loc[
        clean_x_train[col] > 8,
        col
    ] = 8

    clean_x_test.loc[
        clean_x_test[col] > 8,
        col
    ] = 8


# Previously investigated Williamtown evaporation anomaly.
# Use the TRAINING evaporation median as the replacement value.
williamtown_train = (
    (clean_x_train["Location"] == "Williamtown")
    & (clean_x_train["Evaporation"] == 145.0)
)

williamtown_test = (
    (clean_x_test["Location"] == "Williamtown")
    & (clean_x_test["Evaporation"] == 145.0)
)

clean_x_train.loc[
    williamtown_train,
    "Evaporation"
] = train_medians["Evaporation"]

clean_x_test.loc[
    williamtown_test,
    "Evaporation"
] = train_medians["Evaporation"]

print(
    "Williamtown anomaly in train:",
    williamtown_train.sum()
)

print(
    "Williamtown anomaly in test:",
    williamtown_test.sum()
)

# ============================================================
# DATE FEATURES
# ============================================================

clean_x_train["Date"] = pd.to_datetime(
    clean_x_train["Date"]
)

clean_x_test["Date"] = pd.to_datetime(
    clean_x_test["Date"]
)

for data in [clean_x_train, clean_x_test]:
    data["Year"] = data["Date"].dt.year
    data["Month"] = data["Date"].dt.month
    data["Day"] = data["Date"].dt.day

    data.drop(
        columns=["Date"],
        inplace=True
    )


# ============================================================
# RAIN TODAY BINARY ENCODING
# ============================================================

rain_today_map = {
    "No": 0,
    "Yes": 1
}

clean_x_train["RainToday"] = (
    clean_x_train["RainToday"]
    .map(rain_today_map)
)

clean_x_test["RainToday"] = (
    clean_x_test["RainToday"]
    .map(rain_today_map)
)


# ============================================================
# WIND DIRECTION SINE/COSINE ENCODING
# ============================================================

wind_angles = {
    "N": 0,
    "NNE": 22.5,
    "NE": 45,
    "ENE": 67.5,
    "E": 90,
    "ESE": 112.5,
    "SE": 135,
    "SSE": 157.5,
    "S": 180,
    "SSW": 202.5,
    "SW": 225,
    "WSW": 247.5,
    "W": 270,
    "WNW": 292.5,
    "NW": 315,
    "NNW": 337.5
}

wind_columns = [
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm"
]

for col in wind_columns:

    train_angle = clean_x_train[col].map(
        wind_angles
    )

    test_angle = clean_x_test[col].map(
        wind_angles
    )

    clean_x_train[col + "_sin"] = np.sin(
        np.radians(train_angle)
    )

    clean_x_train[col + "_cos"] = np.cos(
        np.radians(train_angle)
    )

    clean_x_test[col + "_sin"] = np.sin(
        np.radians(test_angle)
    )

    clean_x_test[col + "_cos"] = np.cos(
        np.radians(test_angle)
    )


clean_x_train = clean_x_train.drop(
    columns=wind_columns
)

clean_x_test = clean_x_test.drop(
    columns=wind_columns
)

Williamtown anomaly in train: 1
Williamtown anomaly in test: 0


In [11]:
# ============================================================
# LOCATION ENCODING
# FIT MAPPING ON TRAINING DATA ONLY
# ============================================================

train_locations = sorted(
    clean_x_train["Location"].unique()
)

location_map = {
    location: number
    for number, location in enumerate(train_locations)
}

clean_x_train["Location"] = (
    clean_x_train["Location"]
    .map(location_map)
)

clean_x_test["Location"] = (
    clean_x_test["Location"]
    .map(location_map)
)


# check whether test contained any unseen locations
print(
    "Unknown test locations:",
    clean_x_test["Location"].isna().sum()
)

print(
    "Number of training locations:",
    len(location_map)
)

Unknown test locations: 0
Number of training locations: 49


In [12]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("Train shape:", clean_x_train.shape)
print("Test shape:", clean_x_test.shape)

print(
    "\nTraining missing values:",
    clean_x_train.isna().sum().sum()
)

print(
    "Testing missing values:",
    clean_x_test.isna().sum().sum()
)

print(
    "\nNon-numeric training columns:",
    clean_x_train
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print(
    "\nNon-numeric testing columns:",
    clean_x_test
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

print("\nColumns:")
print(clean_x_train.columns.tolist())

Train shape: (113754, 27)
Test shape: (28439, 27)

Training missing values: 0
Testing missing values: 0

Non-numeric training columns: []

Non-numeric testing columns: []

Columns:
['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'Year', 'Month', 'Day', 'WindGustDir_sin', 'WindGustDir_cos', 'WindDir9am_sin', 'WindDir9am_cos', 'WindDir3pm_sin', 'WindDir3pm_cos']


In [13]:
# ============================================================
# FINAL LEAKAGE-FREE DATASET CHECKS
# ============================================================

expected_columns = [
    "Location",
    "MinTemp",
    "MaxTemp",
    "Rainfall",
    "Evaporation",
    "Sunshine",
    "WindGustSpeed",
    "WindSpeed9am",
    "WindSpeed3pm",
    "Humidity9am",
    "Humidity3pm",
    "Pressure9am",
    "Pressure3pm",
    "Cloud9am",
    "Cloud3pm",
    "Temp9am",
    "Temp3pm",
    "RainToday",
    "Year",
    "Month",
    "Day",
    "WindGustDir_sin",
    "WindGustDir_cos",
    "WindDir9am_sin",
    "WindDir9am_cos",
    "WindDir3pm_sin",
    "WindDir3pm_cos"
]

print(
    "Training columns correct:",
    clean_x_train.columns.tolist() == expected_columns
)

print(
    "Testing columns correct:",
    clean_x_test.columns.tolist() == expected_columns
)

print(
    "Training X/Y indices aligned:",
    clean_x_train.index.equals(clean_y_train.index)
)

print(
    "Testing X/Y indices aligned:",
    clean_x_test.index.equals(clean_y_test.index)
)

print(
    "Training rows:",
    len(clean_x_train),
    len(clean_y_train)
)

print(
    "Testing rows:",
    len(clean_x_test),
    len(clean_y_test)
)

Training columns correct: True
Testing columns correct: True
Training X/Y indices aligned: True
Testing X/Y indices aligned: True
Training rows: 113754 113754
Testing rows: 28439 28439


In [14]:
# ============================================================
# LOCK FINAL LEAKAGE-FREE CLEANED DATASETS
# ============================================================

x_train_clean = clean_x_train.reset_index(drop=True).copy()
x_test_clean = clean_x_test.reset_index(drop=True).copy()

y_train_clean = clean_y_train.reset_index(drop=True).copy()
y_test_clean = clean_y_test.reset_index(drop=True).copy()

print("Final cleaned training shape:", x_train_clean.shape)
print("Final cleaned testing shape:", x_test_clean.shape)

print("\nTraining target:")
print(y_train_clean.value_counts())

print("\nTesting target:")
print(y_test_clean.value_counts())

Final cleaned training shape: (113754, 27)
Final cleaned testing shape: (28439, 27)

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Testing target:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [15]:
from pathlib import Path

data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

clean_train_export = x_train_clean.copy()
clean_train_export["RainTomorrow"] = y_train_clean.to_numpy()

clean_test_export = x_test_clean.copy()
clean_test_export["RainTomorrow"] = y_test_clean.to_numpy()

clean_train_export.to_csv(
    data_dir / "weatherAUS_clean_train.csv",
    index=False
)

clean_test_export.to_csv(
    data_dir / "weatherAUS_clean_test.csv",
    index=False
)

print("Saved leakage-free train and test datasets.")
print("Train export:", clean_train_export.shape)
print("Test export:", clean_test_export.shape)

Saved leakage-free train and test datasets.
Train export: (113754, 28)
Test export: (28439, 28)


# **Cleaned eLCS**

In [20]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from skeLCS.eLCS import eLCS

In [17]:
discrete_columns = [
    "Location",
    "RainToday"
]

discrete_attributes = np.array([
    x_train_clean.columns.get_loc(col)
    for col in discrete_columns
])

print("Discrete columns:")
print(discrete_columns)

print("\nDiscrete attribute indices:")
print(discrete_attributes)

print("\nTraining shape:")
print(x_train_clean.shape)

print("\nTesting shape:")
print(x_test_clean.shape)

Discrete columns:
['Location', 'RainToday']

Discrete attribute indices:
[ 0 17]

Training shape:
(113754, 27)

Testing shape:
(28439, 27)


In [21]:
clean_elcs = eLCS(
    learning_iterations=len(x_train_clean),
    N=10000,
    discrete_attribute_limit="d",
    specified_attributes=discrete_attributes,
    random_state=42
)

print("Learning iterations:", len(x_train_clean))
print("Population limit N:", 10000)

_ = clean_elcs.fit(
    x_train_clean.to_numpy(),
    y_train_clean.to_numpy()
)

print("Cleaned eLCS training complete.")

Learning iterations: 113754
Population limit N: 10000
Cleaned eLCS training complete.


In [19]:
clean_elcs_predictions = clean_elcs.predict(
    x_test_clean.to_numpy()
)

print("Prediction worked.")
print(len(clean_elcs_predictions))

Prediction worked.
28439


In [22]:
clean_elcs_predictions = clean_elcs.predict(
    x_test_clean.to_numpy()
)

clean_accuracy = accuracy_score(
    y_test_clean,
    clean_elcs_predictions
)

clean_balanced_accuracy = balanced_accuracy_score(
    y_test_clean,
    clean_elcs_predictions
)

print("Accuracy:", clean_accuracy)

print(
    "Balanced Accuracy:",
    clean_balanced_accuracy
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_clean,
        clean_elcs_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_clean,
        clean_elcs_predictions
    )
)

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

clean_results = pd.DataFrame({
    "row_id": clean_x_test.index.to_numpy(),
    "actual": y_test_clean.to_numpy(),
    "prediction": clean_elcs_predictions
})

clean_results.to_csv(
    results_dir / "clean_elcs_predictions.csv",
    index=False
)

print("Saved clean eLCS predictions.")
clean_elcs.export_final_rule_population(
    x_train_clean.columns.values,
    "RainTomorrow",
    filename=str(
        results_dir / "clean_elcs_rules.csv"
    ),
    DCAL=False
)

clean_elcs.export_iteration_tracking_data(
    str(
        results_dir / "clean_elcs_iterations.csv"
    )
)

Accuracy: 0.8225324378494321
Balanced Accuracy: 0.6197166993701033

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.99      0.90     22064
           1       0.85      0.25      0.39      6375

    accuracy                           0.82     28439
   macro avg       0.84      0.62      0.64     28439
weighted avg       0.83      0.82      0.78     28439


Confusion Matrix:
[[21785   279]
 [ 4768  1607]]
Saved clean eLCS predictions.


# **Improved eLCS**

In [23]:
# ============================================================
# IMPROVED eLCS - 2:1 PARTIAL BALANCING
# ============================================================

improved_train = x_train_clean.copy()
improved_train["RainTomorrow"] = y_train_clean.to_numpy()

no_rain_train = improved_train[
    improved_train["RainTomorrow"] == 0
].copy()

rain_train = improved_train[
    improved_train["RainTomorrow"] == 1
].copy()

print("Original training distribution:")
print(improved_train["RainTomorrow"].value_counts())

# keep ALL rainy examples
rain_sample = rain_train.copy()

# keep twice as many No-rain examples
no_rain_sample = no_rain_train.sample(
    n=len(rain_sample) * 2,
    random_state=42
)

balanced_train = pd.concat(
    [
        no_rain_sample,
        rain_sample
    ],
    ignore_index=True
)

balanced_train = balanced_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

x_train_improved = balanced_train.drop(
    columns=["RainTomorrow"]
)

y_train_improved = balanced_train[
    "RainTomorrow"
].astype(int)

print("\nImproved training distribution:")
print(y_train_improved.value_counts())

print("\nImproved training shape:")
print(x_train_improved.shape)

print("\nUntouched test distribution:")
print(y_test_clean.value_counts())

Original training distribution:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64

Improved training distribution:
RainTomorrow
0    51004
1    25502
Name: count, dtype: int64

Improved training shape:
(76506, 27)

Untouched test distribution:
RainTomorrow
0    22064
1     6375
Name: count, dtype: int64


In [24]:
improved_discrete_attributes = np.array([
    x_train_improved.columns.get_loc("Location"),
    x_train_improved.columns.get_loc("RainToday")
])

improved_elcs = eLCS(
    learning_iterations=len(x_train_improved),
    N=10000,
    discrete_attribute_limit="d",
    specified_attributes=improved_discrete_attributes,
    random_state=42
)

print("Learning iterations:", len(x_train_improved))
print("N:", 10000)

Learning iterations: 76506
N: 10000


In [25]:
# ============================================================
# TRAIN IMPROVED eLCS
# ============================================================

_ = improved_elcs.fit(
    x_train_improved.to_numpy(),
    y_train_improved.to_numpy()
)

print("Improved eLCS training complete.")
# ============================================================
# PREDICT ON ORIGINAL TEST SET
# ============================================================

improved_elcs_predictions = improved_elcs.predict(
    x_test_clean.to_numpy()
)

print("Number of predictions:", len(improved_elcs_predictions))
print("Number of test rows:", len(y_test_clean))

Improved eLCS training complete.
Number of predictions: 28439
Number of test rows: 28439


In [26]:
# ============================================================
# EVALUATE IMPROVED eLCS
# ============================================================

improved_accuracy = accuracy_score(
    y_test_clean,
    improved_elcs_predictions
)

improved_balanced_accuracy = balanced_accuracy_score(
    y_test_clean,
    improved_elcs_predictions
)

print("Accuracy:", improved_accuracy)

print(
    "Balanced Accuracy:",
    improved_balanced_accuracy
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_clean,
        improved_elcs_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_clean,
        improved_elcs_predictions
    )
)

Accuracy: 0.8409578395864834
Balanced Accuracy: 0.7106731398143014

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.95      0.90     22064
           1       0.72      0.47      0.57      6375

    accuracy                           0.84     28439
   macro avg       0.79      0.71      0.74     28439
weighted avg       0.83      0.84      0.83     28439


Confusion Matrix:
[[20891  1173]
 [ 3350  3025]]


In [28]:
# ============================================================
# SAVE IMPROVED eLCS PREDICTIONS
# ============================================================

from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(
    parents=True,
    exist_ok=True
)

test_row_ids = clean_x_test.index.to_numpy()

assert len(test_row_ids) == len(y_test_clean)
assert len(test_row_ids) == len(improved_elcs_predictions)

improved_results = pd.DataFrame({
    "row_id": test_row_ids,
    "actual": y_test_clean.to_numpy(),
    "prediction": improved_elcs_predictions
})

improved_results.to_csv(
    results_dir / "improved_elcs_predictions.csv",
    index=False
)

print("Saved improved eLCS predictions.")
print(improved_results.head())

# ============================================================
# EXPORT FINAL RULE POPULATION
# ============================================================

improved_elcs.export_final_rule_population(
    x_train_improved.columns.values,
    "RainTomorrow",
    filename=str(
        results_dir / "improved_elcs_rules.csv"
    ),
    DCAL=False
)

print("Saved improved eLCS rules.")
# ============================================================
# EXPORT ITERATION TRACKING
# ============================================================

improved_elcs.export_iteration_tracking_data(
    str(
        results_dir / "improved_elcs_iterations.csv"
    )
)

print("Saved improved eLCS iteration data.")

# ============================================================
# FINAL IMPROVED eLCS CHECK
# ============================================================

print(
    "Predictions file exists:",
    (
        results_dir
        / "improved_elcs_predictions.csv"
    ).exists()
)

print(
    "Rules file exists:",
    (
        results_dir
        / "improved_elcs_rules.csv"
    ).exists()
)

print(
    "Iterations file exists:",
    (
        results_dir
        / "improved_elcs_iterations.csv"
    ).exists()
)

Saved improved eLCS predictions.
   row_id  actual  prediction
0   48872       0           0
1  100864       1           1
2  123233       0           0
3    7228       0           0
4    3948       0           0
Saved improved eLCS rules.
Saved improved eLCS iteration data.
Predictions file exists: True
Rules file exists: True
Iterations file exists: True


**Random Forest**


In [39]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training shape:", x_train_clean.shape)
print("Testing shape:", x_test_clean.shape)
print("\nTraining target:")
print(y_train_clean.value_counts())

Training shape: (113754, 27)
Testing shape: (28439, 27)

Training target:
RainTomorrow
0    88252
1    25502
Name: count, dtype: int64


In [40]:
_ = rf_model.fit(
    x_train_clean,
    y_train_clean
)

rf_train_predictions = rf_model.predict(
    x_train_clean
)

rf_predictions = rf_model.predict(
    x_test_clean
)

print("Random Forest training complete.")

Random Forest training complete.


In [41]:
rf_train_accuracy = accuracy_score(
    y_train_clean,
    rf_train_predictions
)

rf_accuracy = accuracy_score(
    y_test_clean,
    rf_predictions
)

rf_balanced_accuracy = balanced_accuracy_score(
    y_test_clean,
    rf_predictions
)

print(
    "Training Accuracy:",
    rf_train_accuracy
)

print(
    "Testing Accuracy:",
    rf_accuracy
)

print(
    "Balanced Accuracy:",
    rf_balanced_accuracy
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_clean,
        rf_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_clean,
        rf_predictions
    )
)

Training Accuracy: 0.9798336761784201
Testing Accuracy: 0.8483420654734696
Balanced Accuracy: 0.7957966272803538

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.89      0.90     22064
           1       0.65      0.70      0.67      6375

    accuracy                           0.85     28439
   macro avg       0.78      0.80      0.79     28439
weighted avg       0.85      0.85      0.85     28439


Confusion Matrix:
[[19660  2404]
 [ 1909  4466]]


In [42]:
rf_results = pd.DataFrame({
    "row_id": clean_x_test.index.to_numpy(),
    "actual": y_test_clean.to_numpy(),
    "prediction": rf_predictions
})

rf_results.to_csv(
    results_dir / "random_forest_predictions.csv",
    index=False
)

print("Random Forest predictions saved.")
print(rf_results.head())

Random Forest predictions saved.
   row_id  actual  prediction
0   48872       0           0
1  100864       1           1
2  123233       0           0
3    7228       0           0
4    3948       0           0


In [43]:
print(
    "RF and cleaned eLCS test rows identical:",
    np.array_equal(
        rf_results["row_id"].to_numpy(),
        clean_results["row_id"].to_numpy()
    )
)

print(
    "RF and cleaned eLCS targets identical:",
    np.array_equal(
        rf_results["actual"].to_numpy(),
        clean_results["actual"].to_numpy()
    )
)

RF and cleaned eLCS test rows identical: True
RF and cleaned eLCS targets identical: True


**XGBoost**

In [49]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# calculate class weighting using TRAINING data only
no_rain_count = (y_train_clean == 0).sum()
rain_count = (y_train_clean == 1).sum()

scale_pos_weight = no_rain_count / rain_count

print("No Rain:", no_rain_count)
print("Rain:", rain_count)
print("Scale pos weight:", scale_pos_weight)

No Rain: 88252
Rain: 25502
Scale pos weight: 3.4605913261704964


In [50]:
xgb_model = XGBClassifier(
    n_estimators=5000,
    learning_rate=1,
    max_depth=10,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

print("Training shape:", x_train_clean.shape)
print("Testing shape:", x_test_clean.shape)

Training shape: (113754, 27)
Testing shape: (28439, 27)


In [51]:
_ = xgb_model.fit(
    x_train_clean,
    y_train_clean
)

xgb_train_predictions = xgb_model.predict(
    x_train_clean
)

xgb_predictions = xgb_model.predict(
    x_test_clean
)

print("XGBoost training complete.")

XGBoost training complete.


In [52]:
xgb_train_accuracy = accuracy_score(
    y_train_clean,
    xgb_train_predictions
)

xgb_accuracy = accuracy_score(
    y_test_clean,
    xgb_predictions
)

xgb_balanced_accuracy = balanced_accuracy_score(
    y_test_clean,
    xgb_predictions
)

print(
    "Training Accuracy:",
    xgb_train_accuracy
)

print(
    "Testing Accuracy:",
    xgb_accuracy
)

print(
    "Balanced Accuracy:",
    xgb_balanced_accuracy
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_clean,
        xgb_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_clean,
        xgb_predictions
    )
)

xgb_results = pd.DataFrame({
    "row_id": clean_x_test.index.to_numpy(),
    "actual": y_test_clean.to_numpy(),
    "prediction": xgb_predictions
})

xgb_results.to_csv(
    results_dir / "xgboost_predictions.csv",
    index=False
)

print("XGBoost predictions saved.")
print(xgb_results.head())

Training Accuracy: 1.0
Testing Accuracy: 0.8566053658708113
Balanced Accuracy: 0.7751889832074962

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.92      0.91     22064
           1       0.70      0.63      0.66      6375

    accuracy                           0.86     28439
   macro avg       0.80      0.78      0.79     28439
weighted avg       0.85      0.86      0.85     28439


Confusion Matrix:
[[20360  1704]
 [ 2374  4001]]
XGBoost predictions saved.
   row_id  actual  prediction
0   48872       0           0
1  100864       1           1
2  123233       0           1
3    7228       0           0
4    3948       0           0


In [53]:
print(
    "XGBoost and cleaned eLCS test rows identical:",
    np.array_equal(
        xgb_results["row_id"].to_numpy(),
        clean_results["row_id"].to_numpy()
    )
)

print(
    "XGBoost and cleaned eLCS targets identical:",
    np.array_equal(
        xgb_results["actual"].to_numpy(),
        clean_results["actual"].to_numpy()
    )
)

XGBoost and cleaned eLCS test rows identical: True
XGBoost and cleaned eLCS targets identical: True


**Linear Regression**


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Location is categorical even though it is currently stored as integers
location_columns = ["Location"]

# Scale all remaining numerical predictors
logistic_numeric_columns = [
    col for col in x_train_clean.columns
    if col != "Location"
]

logistic_preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            location_columns
        ),
        (
            "numeric",
            StandardScaler(),
            logistic_numeric_columns
        )
    ]
)

logistic_model = Pipeline([
    (
        "preprocessor",
        logistic_preprocessor
    ),
    (
        "logistic_regression",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )
])

print("Training shape:", x_train_clean.shape)
print("Testing shape:", x_test_clean.shape)

Training shape: (113754, 27)
Testing shape: (28439, 27)


In [56]:
_ = logistic_model.fit(
    x_train_clean,
    y_train_clean
)

logistic_train_predictions = logistic_model.predict(
    x_train_clean
)

logistic_predictions = logistic_model.predict(
    x_test_clean
)

print("Logistic Regression training complete.")

logistic_train_accuracy = accuracy_score(
    y_train_clean,
    logistic_train_predictions
)

logistic_accuracy = accuracy_score(
    y_test_clean,
    logistic_predictions
)

logistic_balanced_accuracy = balanced_accuracy_score(
    y_test_clean,
    logistic_predictions
)

print(
    "Training Accuracy:",
    logistic_train_accuracy
)

print(
    "Testing Accuracy:",
    logistic_accuracy
)

print(
    "Balanced Accuracy:",
    logistic_balanced_accuracy
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_clean,
        logistic_predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_clean,
        logistic_predictions
    )
)

Logistic Regression training complete.
Training Accuracy: 0.7934226488738858
Testing Accuracy: 0.7941559126551566
Balanced Accuracy: 0.788314340456995

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.80      0.86     22064
           1       0.53      0.78      0.63      6375

    accuracy                           0.79     28439
   macro avg       0.73      0.79      0.74     28439
weighted avg       0.84      0.79      0.81     28439


Confusion Matrix:
[[17627  4437]
 [ 1417  4958]]


In [57]:
logistic_results = pd.DataFrame({
    "row_id": clean_x_test.index.to_numpy(),
    "actual": y_test_clean.to_numpy(),
    "prediction": logistic_predictions
})

logistic_results.to_csv(
    results_dir / "logistic_regression_predictions.csv",
    index=False
)

print("Logistic Regression predictions saved.")
print(logistic_results.head())

print(
    "Logistic Regression and cleaned eLCS test rows identical:",
    np.array_equal(
        logistic_results["row_id"].to_numpy(),
        clean_results["row_id"].to_numpy()
    )
)

print(
    "Logistic Regression and cleaned eLCS targets identical:",
    np.array_equal(
        logistic_results["actual"].to_numpy(),
        clean_results["actual"].to_numpy()
    )
)

Logistic Regression predictions saved.
   row_id  actual  prediction
0   48872       0           0
1  100864       1           1
2  123233       0           1
3    7228       0           0
4    3948       0           0
Logistic Regression and cleaned eLCS test rows identical: True
Logistic Regression and cleaned eLCS targets identical: True


In [58]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

results_dir = Path("../results")

prediction_files = {
    "Raw eLCS": "raw_elcs_predictions.csv",
    "Cleaned eLCS": "clean_elcs_predictions.csv",
    "Improved eLCS": "improved_elcs_predictions.csv",
    "Random Forest": "random_forest_predictions.csv",
    "XGBoost": "xgboost_predictions.csv",
    "Logistic Regression": "logistic_regression_predictions.csv"
}

prediction_data = {}

for model_name, filename in prediction_files.items():
    prediction_data[model_name] = pd.read_csv(
        results_dir / filename
    )

    print(
        model_name,
        prediction_data[model_name].shape
    )

Raw eLCS (28439, 3)
Cleaned eLCS (28439, 3)
Improved eLCS (28439, 3)
Random Forest (28439, 3)
XGBoost (28439, 3)
Logistic Regression (28439, 3)


In [59]:
reference_rows = prediction_data[
    "Cleaned eLCS"
]["row_id"].to_numpy()

reference_actual = prediction_data[
    "Cleaned eLCS"
]["actual"].to_numpy()

for model_name, data in prediction_data.items():

    same_rows = np.array_equal(
        reference_rows,
        data["row_id"].to_numpy()
    )

    same_targets = np.array_equal(
        reference_actual,
        data["actual"].to_numpy()
    )

    print(
        model_name,
        "| Same rows:",
        same_rows,
        "| Same targets:",
        same_targets
    )

Raw eLCS | Same rows: True | Same targets: True
Cleaned eLCS | Same rows: True | Same targets: True
Improved eLCS | Same rows: True | Same targets: True
Random Forest | Same rows: True | Same targets: True
XGBoost | Same rows: True | Same targets: True
Logistic Regression | Same rows: True | Same targets: True


In [60]:
comparison_rows = []

for model_name, data in prediction_data.items():

    actual = data["actual"].astype(int)
    prediction = data["prediction"].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual,
        prediction
    ).ravel()

    comparison_rows.append({
        "Model": model_name,

        "Accuracy": accuracy_score(
            actual,
            prediction
        ),

        "Balanced Accuracy": balanced_accuracy_score(
            actual,
            prediction
        ),

        "Rain Precision": precision_score(
            actual,
            prediction,
            pos_label=1
        ),

        "Rain Recall": recall_score(
            actual,
            prediction,
            pos_label=1
        ),

        "Rain F1": f1_score(
            actual,
            prediction,
            pos_label=1
        ),

        "True Positives": tp,
        "False Positives": fp,
        "True Negatives": tn,
        "False Negatives": fn
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df = comparison_df.round(4)

print(comparison_df.to_string(index=False))

              Model  Accuracy  Balanced Accuracy  Rain Precision  Rain Recall  Rain F1  True Positives  False Positives  True Negatives  False Negatives
           Raw eLCS    0.8295             0.6384          0.8467       0.2921   0.4343            1862              337           21727             4513
       Cleaned eLCS    0.8225             0.6197          0.8521       0.2521   0.3891            1607              279           21785             4768
      Improved eLCS    0.8410             0.7107          0.7206       0.4745   0.5722            3025             1173           20891             3350
      Random Forest    0.8483             0.7958          0.6501       0.7005   0.6744            4466             2404           19660             1909
            XGBoost    0.8566             0.7752          0.7013       0.6276   0.6624            4001             1704           20360             2374
Logistic Regression    0.7942             0.7883          0.5277       0.7777   0.

In [61]:
comparison_df.to_csv(
    results_dir / "final_model_comparison.csv",
    index=False
)

print("Final model comparison saved.")

Final model comparison saved.


# Statistical Significance Testing

In [67]:
def run_mcnemar(model_a_name, model_b_name):

    model_a = prediction_data[model_a_name]
    model_b = prediction_data[model_b_name]

    assert np.array_equal(
        model_a["row_id"],
        model_b["row_id"]
    )

    assert np.array_equal(
        model_a["actual"],
        model_b["actual"]
    )

    actual = model_a["actual"].to_numpy()

    a_pred = model_a["prediction"].to_numpy()
    b_pred = model_b["prediction"].to_numpy()

    a_correct = a_pred == actual
    b_correct = b_pred == actual

    a_only = np.sum(
        a_correct & ~b_correct
    )

    b_only = np.sum(
        ~a_correct & b_correct
    )

    table = [
        [
            np.sum(a_correct & b_correct),
            a_only
        ],
        [
            b_only,
            np.sum(~a_correct & ~b_correct)
        ]
    ]

    result = mcnemar(
        table,
        exact=False,
        correction=True
    )

    print(
        f"\n{model_a_name} vs {model_b_name}"
    )

    print("Table:", table)
    print(
        f"{model_a_name} only correct:",
        a_only
    )

    print(
        f"{model_b_name} only correct:",
        b_only
    )

    print(
        "Statistic:",
        result.statistic
    )

    print(
        "p-value:",
        result.pvalue
    )

In [68]:
run_mcnemar(
    "Cleaned eLCS",
    "Improved eLCS"
)

run_mcnemar(
    "Improved eLCS",
    "Random Forest"
)


Cleaned eLCS vs Improved eLCS
Table: [[np.int64(22498), np.int64(894)], [np.int64(1418), np.int64(3629)]]
Cleaned eLCS only correct: 894
Improved eLCS only correct: 1418
Statistic: 118.3083910034602
p-value: 1.484209642218513e-27

Improved eLCS vs Random Forest
Table: [[np.int64(22536), np.int64(1380)], [np.int64(1590), np.int64(2933)]]
Improved eLCS only correct: 1380
Random Forest only correct: 1590
Statistic: 14.707407407407407
p-value: 0.00012555214567059567


In [69]:
rules = pd.read_csv(
    "../results/improved_elcs_rules.csv"
)

print("Shape:", rules.shape)
print("\nColumns:")
print(rules.columns.tolist())

print("\nFirst 10 rules:")
print(rules.head(10).to_string())

Shape: (9465, 38)

Columns:
['Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm', 'RainToday', 'Year', 'Month', 'Day', 'WindGustDir_sin', 'WindGustDir_cos', 'WindDir9am_sin', 'WindDir9am_cos', 'WindDir3pm_sin', 'WindDir3pm_cos', 'RainTomorrow', 'Fitness', 'Accuracy', 'Numerosity', 'Avg Match Set Size', 'TimeStamp GA', 'Iteration Initialized', 'Specificity', 'Deletion Probability', 'Correct Count', 'Match Count']

First 10 rules:
  Location                                 MinTemp                                MaxTemp                                Rainfall                 Evaporation                               Sunshine                         WindGustSpeed                            WindSpeed9am                             WindSpeed3pm                Humidity9am               Humidity3pm                        

In [70]:
feature_columns = x_train_improved.columns.tolist()

# Count how many conditions each rule actually specifies.
# "#" means the rule does not care about that feature.
rules["Specified Conditions"] = (
    rules[feature_columns] != "#"
).sum(axis=1)

# Keep rules that have actually been used a reasonable number of times
strong_rules = rules[
    (rules["Match Count"] >= 20)
    & (rules["Accuracy"] >= 0.80)
].copy()

print("Strong candidate rules:", len(strong_rules))

candidate_summary = strong_rules[
    [
        "RainTomorrow",
        "Accuracy",
        "Fitness",
        "Numerosity",
        "Correct Count",
        "Match Count",
        "Specificity",
        "Specified Conditions"
    ]
].sort_values(
    by=[
        "RainTomorrow",
        "Accuracy",
        "Match Count"
    ],
    ascending=[
        True,
        False,
        False
    ]
)

print(candidate_summary.head(30).to_string())

Strong candidate rules: 4019
      RainTomorrow  Accuracy  Fitness  Numerosity  Correct Count  Match Count  Specificity  Specified Conditions
1461           0.0       1.0      1.0           1            151          151     0.222222                     6
1560           0.0       1.0      1.0           2            143          143     0.185185                     5
3182           0.0       1.0      1.0           2            142          142     0.370370                    10
601            0.0       1.0      1.0           1            136          136     0.407407                    11
2075           0.0       1.0      1.0           1            128          128     0.444444                    12
2548           0.0       1.0      1.0           2            123          123     0.333333                     9
2326           0.0       1.0      1.0           1            122          122     0.555556                    15
4060           0.0       1.0      1.0           1            119   

In [73]:
rain_rule_candidates = strong_rules[
    strong_rules["RainTomorrow"] == 1
].sort_values(
    by=[
        "Match Count",
        "Accuracy"
    ],
    ascending=[
        False,
        False
    ]
)

print(
    rain_rule_candidates[
        [
            "Accuracy",
            "Fitness",
            "Numerosity",
            "Correct Count",
            "Match Count",
            "Specificity",
            "Specified Conditions"
        ]
    ].head(20).to_string()
)

      Accuracy   Fitness  Numerosity  Correct Count  Match Count  Specificity  Specified Conditions
56    0.847404  0.436970           1           4798         5662     0.222222                     6
530   0.826053  0.384626           1           2099         2541     0.222222                     6
92    0.897959  0.583825           1           1980         2205     0.222222                     6
66    0.906636  0.612583           1           1981         2185     0.296296                     8
296   0.857939  0.464816           1           1848         2154     0.222222                     6
1375  0.810433  0.349612           1           1740         2147     0.111111                     3
548   0.838891  0.415460           1           1786         2129     0.296296                     8
120   0.815870  0.361497           1           1604         1966     0.222222                     6
786   0.865047  0.484392           1           1673         1934     0.148148                     4


In [75]:
selected_rule_ids = [
    56,
    92,
    1461
]

selected_rules = rules.loc[
    selected_rule_ids
].copy()

for rule_id, rule in selected_rules.iterrows():

    print("=" * 70)
    print("RULE:", rule_id)

    print(
        "Prediction:",
        int(rule["RainTomorrow"])
    )

    print(
        "Accuracy:",
        round(rule["Accuracy"], 4)
    )

    print(
        "Fitness:",
        round(rule["Fitness"], 4)
    )

    print(
        "Numerosity:",
        int(rule["Numerosity"])
    )

    print(
        "Correct Count:",
        int(rule["Correct Count"])
    )

    print(
        "Match Count:",
        int(rule["Match Count"])
    )

    print(
        "Specificity:",
        round(rule["Specificity"], 4)
    )

    print("\nSpecified conditions:")

    for feature in feature_columns:

        value = rule[feature]

        if str(value) != "#":
            print(
                feature,
                "→",
                value
            )

    print()

RULE: 56
Prediction: 1
Accuracy: 0.8474
Fitness: 0.437
Numerosity: 1
Correct Count: 4798
Match Count: 5662
Specificity: 0.2222

Specified conditions:
Humidity9am → 49.665,127.66
Humidity3pm → 73.22,125.215
Pressure9am → 979.682,1062.6099792948844
Cloud9am → 4.16,9.2
RainToday → 1.0
WindDir3pm_cos → -1.1471439310134237,1.1771067811865474

RULE: 92
Prediction: 1
Accuracy: 0.898
Fitness: 0.5838
Numerosity: 1
Correct Count: 1980
Match Count: 2205
Specificity: 0.2222

Specified conditions:
Evaporation → -5.3500000000000005,14.95
WindSpeed3pm → -17.099999999999998,35.099999999999994
Humidity3pm → 78.13,106.195
Pressure3pm → 987.751,1015.125
Cloud9am → 4.48,9.84
Day → -0.25,22.25

RULE: 1461
Prediction: 0
Accuracy: 1.0
Fitness: 1.0
Numerosity: 1
Correct Count: 151
Match Count: 151
Specificity: 0.2222

Specified conditions:
Evaporation → -13.064,27.536000000000005
Sunshine → 4.711,11.289
Humidity3pm → -23.745962342240986,28.02672492278412
Pressure3pm → 1017.7760000000001,1052.522
Temp3pm → 4.2

In [74]:
no_rain_rule_candidates = strong_rules[
    strong_rules["RainTomorrow"] == 0
].sort_values(
    by=[
        "Match Count",
        "Accuracy"
    ],
    ascending=[
        False,
        False
    ]
)

print(
    no_rain_rule_candidates[
        [
            "Accuracy",
            "Fitness",
            "Numerosity",
            "Correct Count",
            "Match Count",
            "Specificity",
            "Specified Conditions"
        ]
    ].head(10).to_string()
)

      Accuracy   Fitness  Numerosity  Correct Count  Match Count  Specificity  Specified Conditions
49    0.833769  0.402929           1          16276        19521     0.185185                     5
506   0.845679  0.432541           1          13689        16187     0.259259                     7
1074  0.814809  0.359152           1          10080        12371     0.222222                     6
317   0.817224  0.364508           1          10011        12250     0.222222                     6
1195  0.800033  0.327748           1           9694        12117     0.185185                     5
1513  0.805087  0.338232           1           8831        10969     0.296296                     8
470   0.819921  0.370560           1           8874        10823     0.185185                     5
1802  0.817492  0.365105           1           7973         9753     0.148148                     4
300   0.918269  0.652905           1           8786         9568     0.185185                     5


In [76]:
rule = rules.loc[300]

print("RULE 300")
print("Prediction:", int(rule["RainTomorrow"]))
print("Accuracy:", rule["Accuracy"])
print("Correct Count:", int(rule["Correct Count"]))
print("Match Count:", int(rule["Match Count"]))

print("\nSpecified conditions:")

for feature in feature_columns:
    if str(rule[feature]) != "#":
        print(feature, "→", rule[feature])

RULE 300
Prediction: 0
Accuracy: 0.9182692307692308
Correct Count: 8786
Match Count: 9568

Specified conditions:
Rainfall → -113.155,113.155
Evaporation → -9.852,36.050000000000004
WindGustSpeed → 13.069999999999997,44.705
Humidity3pm → -23.745962342240986,51.660000000000004
RainToday → 0.0


## Selected eLCS Rules for Interpretation

In [77]:
selected_rule_summary = pd.DataFrame([
    {
        "Rule": 56,
        "Prediction": "Rain",
        "Accuracy": round(rules.loc[56, "Accuracy"], 4),
        "Correct Count": int(rules.loc[56, "Correct Count"]),
        "Match Count": int(rules.loc[56, "Match Count"]),
        "Specified Conditions": int(rules.loc[56, "Specified Conditions"]),
        "Main Pattern": "RainToday = Yes, high afternoon humidity and cloud"
    },
    {
        "Rule": 92,
        "Prediction": "Rain",
        "Accuracy": round(rules.loc[92, "Accuracy"], 4),
        "Correct Count": int(rules.loc[92, "Correct Count"]),
        "Match Count": int(rules.loc[92, "Match Count"]),
        "Specified Conditions": int(rules.loc[92, "Specified Conditions"]),
        "Main Pattern": "Very high afternoon humidity, lower pressure and cloud"
    },
    {
        "Rule": 300,
        "Prediction": "No Rain",
        "Accuracy": round(rules.loc[300, "Accuracy"], 4),
        "Correct Count": int(rules.loc[300, "Correct Count"]),
        "Match Count": int(rules.loc[300, "Match Count"]),
        "Specified Conditions": int(rules.loc[300, "Specified Conditions"]),
        "Main Pattern": "RainToday = No, lower afternoon humidity and moderate wind gusts"
    }
])

print(selected_rule_summary.to_string(index=False))

selected_rule_summary.to_csv(
    results_dir / "selected_elcs_rules_summary.csv",
    index=False
)

print("Selected rule summary saved.")

 Rule Prediction  Accuracy  Correct Count  Match Count  Specified Conditions                                                     Main Pattern
   56       Rain    0.8474           4798         5662                     6               RainToday = Yes, high afternoon humidity and cloud
   92       Rain    0.8980           1980         2205                     6           Very high afternoon humidity, lower pressure and cloud
  300    No Rain    0.9183           8786         9568                     5 RainToday = No, lower afternoon humidity and moderate wind gusts
Selected rule summary saved.
